# SEER GBM 预测模型

基于 Cox 比例风险模型构建 GBM 预后预测模型，包括:
1. Cox 回归 (训练/测试集拆分)
2. Time-dependent ROC 曲线 (1/3/5年)
3. 校准曲线
4. 决策曲线分析 (DCA)
5. 列线图 (Nomogram)

变量编码:
- Age: 0=<60, 1=60-69, 2=≥70
- Surgery: 0=No, 1=Yes
- Radiation: 0=No, 1=Yes
- Chemotherapy: 0=No, 1=Yes
- Stage: 0=Localized, 1=Regional, 2=Distant, 3=Unknown/unstaged

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from lifelines import CoxPHFitter
from lifelines import KaplanMeierFitter
from sklearn.metrics import roc_curve, auc
from lifelines.utils import concordance_index

# 全局绘图设置
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.linewidth': 0.8,
    'axes.unicode_minus': False,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.format': 'tiff',
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = ['#5B9BD5', '#ED7D31', '#70AD47', '#FFC000', '#9B59B6']

df = pd.read_csv('cleaned_data.csv')

In [ ]:
# ===== Cox 比例风险模型 =====
# 变量编码: 分类变量转为数值型，纳入 Cox 回归
# Age: <60=0, 60-69=1, ≥70=2 (有序编码)
# Stage: Localized=0, Regional=1, Distant=2, Unknown=3 (有序编码)
# duration_col: Survival months (生存时间，月)
# event_col: Vital status recode (0=删失, 1=死亡)
df['Age'] = df['Age Group'].map({'<60': 0, '60-69': 1, '≥70': 2})
df_cox = df[['Survival months', 'Vital status recode', 'Age',
             'Surgery', 'Radiation', 'Chemotherapy']].copy()
df_cox['Stage'] = df['Stage'].map({'Localized': 0, 'Regional': 1, 'Distant': 2, 'Unknown/unstaged': 3})

# 拆分训练集/测试集 (7:3)
from sklearn.model_selection import train_test_split
train, test = train_test_split(df_cox, test_size=0.3, random_state=42)

# 训练 Cox 模型
cph = CoxPHFitter()
cph.fit(train, duration_col='Survival months', event_col='Vital status recode')
cph.print_summary()

In [ ]:
# ===== Time-dependent ROC 曲线 =====
# 评估模型在不同时间点的区分能力
# y_true: 是否在 t 个月内死亡 (1=是, 0=否)
# y_score: Cox 模型预测的部分风险评分 (partial hazard)
fig, ax = plt.subplots(figsize=(6, 6))

# 预测风险评分
test['risk_score'] = cph.predict_partial_hazard(test)

# 时间点: 1年(12月), 3年(36月), 5年(60月)
time_points = [12, 36, 60]
labels = ['1-year', '3-year', '5-year']

for i, (t, label) in enumerate(zip(time_points, labels)):
    # 真实标签: 在t个月内是否死亡
    y_true = ((test['Survival months'] <= t) & (test['Vital status recode'] == 1)).astype(int)
    
    # 风险评分作为预测值
    y_score = test['risk_score']
    
    # 计算 ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, color=COLORS[i], linewidth=1.5, 
            label=f'{label} (AUC = {roc_auc:.3f})')

# 参考线: 随机猜测
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Time-dependent ROC Curves', fontweight='bold', pad=10)
ax.legend(title='Survival Time', frameon=False)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
fig.savefig('fig13_roc.tiff')
plt.close()

In [ ]:
# ===== 校准曲线 =====
# 比较预测风险 vs 实际事件发生率
# 理想情况: 校准曲线贴近对角线 (y=x)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for i, (t, label) in enumerate(zip(time_points, labels)):
    ax = axes[i]
    
    # 真实标签
    y_true = ((test['Survival months'] <= t) & (test['Vital status recode'] == 1)).astype(int)
    
    # 预测风险
    y_score = test['risk_score']
    
    # 分10组计算每组的平均预测风险和实际事件率
    y_score_binned = pd.cut(y_score, bins=10)
    grouped = pd.DataFrame({'y_true': y_true, 'y_pred': y_score}).groupby(y_score_binned, observed=False).mean()
    
    # 校准曲线
    ax.plot(grouped['y_pred'], grouped['y_true'], 'o-', color=COLORS[i], linewidth=1.5, markersize=4)
    # 理想参考线
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Predicted Risk', fontsize=10)
    ax.set_ylabel('Actual Event Rate', fontsize=10)
    ax.set_title(f'{label} Calibration', fontsize=11, fontweight='bold')

plt.tight_layout()
fig.savefig('fig14_calibration.tiff')
plt.close()

In [ ]:
# ===== 决策曲线分析 (DCA) =====
# 评估模型的临床净获益 (Net Benefit)
# NB = (TP/N) - (FP/N) × (pt/(1-pt))
# pt: 阈值概率，表示患者/医生愿意接受的假阳性代价
# Treat All: 对所有人都预测事件
# Treat None: 对所有人都不预测事件 (NB=0)
fig, ax = plt.subplots(figsize=(6, 6))

y_score = test['risk_score']
y_true_1y = ((test['Survival months'] <= 12) & (test['Vital status recode'] == 1)).astype(int)

# 计算不同阈值下的 Net Benefit
thresholds = np.linspace(0, 0.99, 200)
net_benefits = []
for pt in thresholds:
    y_pred = (y_score > pt).astype(int)
    tp = ((y_pred == 1) & (y_true_1y == 1)).sum()
    fp = ((y_pred == 1) & (y_true_1y == 0)).sum()
    n = len(y_true_1y)
    nb = (tp / n) - (fp / n) * (pt / (1 - pt))
    net_benefits.append(nb)

# Treat All 线: 使用人群患病率
prevalence = y_true_1y.mean()
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))

ax.plot(thresholds, net_benefits, color=COLORS[0], linewidth=1.5, label='Cox Model')
ax.plot(thresholds, nb_all, color='gray', linestyle='--', linewidth=0.8, label='Treat All')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, label='Treat None')

ax.set_xlabel('Threshold Probability', fontsize=11)
ax.set_ylabel('Net Benefit', fontsize=11)
ax.set_title('Decision Curve Analysis (1-year)', fontweight='bold', pad=10)
ax.legend(frameon=False)
ax.set_xlim([0, 0.8])
ax.set_ylim([-0.05, max(net_benefits) * 1.1])
plt.tight_layout()
fig.savefig('fig15_dca.tiff')
plt.close()

In [ ]:
# ===== 列线图 (Nomogram) =====
# 将 Cox 回归结果可视化为临床评分工具
# 每个变量根据其回归系数分配对应的分值 (Points)
# 各变量分值相加得到总分，对应预测的生存概率
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# 列线图参数
variables = ['Age', 'Surgery', 'Radiation', 'Chemotherapy', 'Stage']
titles = ['Age Group', 'Surgery', 'Radiation', 'Chemotherapy', 'Stage']
labels_list = [
    ['<60', '60-69', '≥70'],
    ['No', 'Yes'],
    ['No', 'Yes'],
    ['No', 'Yes'],
    ['Localized', 'Regional', 'Distant', 'Unknown']
]
scores = [(0, 50, 100), (0, 100), (0, 100), (0, 100), (0, 30, 60, 100)]

y_positions = np.linspace(0.1, 0.9, len(variables) + 1)

# Points 标尺 (顶部)
ax.plot([0.1, 0.9], [y_positions[0], y_positions[0]], 'k-', linewidth=1)
for i, score in enumerate(scores[0]):
    x = 0.1 + (i / (len(scores[0]) - 1)) * 0.8
    ax.plot([x, x], [y_positions[0] - 0.02, y_positions[0]], 'k-', linewidth=1)
    ax.text(x, y_positions[0] - 0.05, str(score), ha='center', va='top', fontsize=9)
ax.text(-0.05, y_positions[0], 'Points', ha='right', va='center', fontsize=10, fontweight='bold')

# 各变量标尺
for j, (title, lab, sc) in enumerate(zip(titles, labels_list, scores)):
    y = y_positions[j + 1]
    ax.plot([0.1, 0.9], [y, y], 'k-', linewidth=1)
    
    for i, label in enumerate(lab):
        x = 0.1 + (i / (len(lab) - 1)) * 0.8 if len(lab) > 1 else 0.5
        ax.plot([x, x], [y - 0.02, y], 'k-', linewidth=1)
        ax.text(x, y - 0.05, label, ha='center', va='top', fontsize=9)
    
    ax.text(-0.05, y, title, ha='right', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(-0.3, 1.2)
ax.set_ylim(0, 1)
ax.set_title('Nomogram for Predicting Overall Survival in GBM Patients', 
             fontweight='bold', pad=20, fontsize=12)

plt.tight_layout()
fig.savefig('fig16_nomogram.tiff')
plt.close()